# Paired Bootstrap

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score, roc_auc_score, f1_score,
    recall_score, precision_score, matthews_corrcoef
)

def calc_metric(y, p, metric, thr=None):
    y = np.asarray(y).astype(int)
    p = np.asarray(p, dtype=float)

    if metric == "ap":
        return average_precision_score(y, p)
    elif metric == "auc":
        return roc_auc_score(y, p)
    elif metric == "f1_05":
        return f1_score(y, (p >= 0.5).astype(int), zero_division=0)
    elif metric == "precision_05":
        return precision_score(y, (p >= 0.5).astype(int), zero_division=0)
    elif metric == "recall_05":
        return recall_score(y, (p >= 0.5).astype(int), zero_division=0)
    elif metric == "mcc_05":
        return matthews_corrcoef(y, (p >= 0.5).astype(int))
    elif metric == "f1_opt":
        return f1_score(y, (p >= thr).astype(int), zero_division=0)
    elif metric == "precision_opt":
        return precision_score(y, (p >= thr).astype(int), zero_division=0)
    elif metric == "recall_opt":
        return recall_score(y, (p >= thr).astype(int), zero_division=0)
    elif metric == "mcc_opt":
        return matthews_corrcoef(y, (p >= thr).astype(int))
    else:
        raise ValueError(f"Unsupported metric: {metric}")

def paired_bootstrap(
    y,
    p_a,
    p_b,
    metric="ap",
    thr_a=None,
    thr_b=None,
    n_boot=5000,
    stratified=True,
    seed=42
):
    y = np.asarray(y).astype(int)
    p_a = np.asarray(p_a, dtype=float)
    p_b = np.asarray(p_b, dtype=float)

    assert len(y) == len(p_a) == len(p_b)

    rng = np.random.default_rng(seed)
    n = len(y)

    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    diffs = np.empty(n_boot, dtype=float)

    for b in range(n_boot):
        if stratified:
            boot_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
            boot_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
            idx = np.concatenate([boot_pos, boot_neg])
        else:
            idx = rng.choice(np.arange(n), size=n, replace=True)

        y_b = y[idx]
        p_a_b = p_a[idx]
        p_b_b = p_b[idx]

        m_a = calc_metric(y_b, p_a_b, metric=metric, thr=thr_a)
        m_b = calc_metric(y_b, p_b_b, metric=metric, thr=thr_b)
        diffs[b] = m_b - m_a

    point_a = calc_metric(y, p_a, metric=metric, thr=thr_a)
    point_b = calc_metric(y, p_b, metric=metric, thr=thr_b)
    point_diff = point_b - point_a

    ci_low, ci_high = np.quantile(diffs, [0.025, 0.975])

    r_left = np.sum(diffs <= 0)
    r_right = np.sum(diffs >= 0)
    r = min(r_left, r_right)

    p_value = min(1.0, 2 * (r + 1) / (n_boot + 1))

    return {
        "metric": metric,
        "model_a": point_a,
        "model_b": point_b,
        "diff_b_minus_a": point_diff,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "n_boot": n_boot
    }

def run_paired_bootstrap_from_csv(
    csv_a,
    csv_b,
    model_a_name="model_a",
    model_b_name="model_b",
    n_boot=5000,
    stratified=True,
    seed=42
):
    a = pd.read_csv(csv_a).rename(columns={
        "pred_prob": "pred_prob_a",
        "threshold_opt": "threshold_opt_a"
    })
    b = pd.read_csv(csv_b).rename(columns={
        "pred_prob": "pred_prob_b",
        "threshold_opt": "threshold_opt_b"
    })

    pair_df = a.merge(
        b,
        on=["sno_hashed", "y_true"],
        how="inner",
        validate="one_to_one"
    ).copy()

    metrics = [
        "ap", "auc",
        "f1_05", "precision_05", "recall_05", "mcc_05",
        "f1_opt", "precision_opt", "recall_opt", "mcc_opt"
    ]

    results = []
    thr_a = float(pair_df["threshold_opt_a"].iloc[0])
    thr_b = float(pair_df["threshold_opt_b"].iloc[0])

    for metric in metrics:
        res = paired_bootstrap(
            y=pair_df["y_true"].values,
            p_a=pair_df["pred_prob_a"].values,
            p_b=pair_df["pred_prob_b"].values,
            metric=metric,
            thr_a=thr_a,
            thr_b=thr_b,
            n_boot=n_boot,
            stratified=stratified,
            seed=seed
        )
        res["model_a_name"] = model_a_name
        res["model_b_name"] = model_b_name
        res["n_common"] = len(pair_df)
        results.append(res)

    results_df = pd.DataFrame(results)
    return pair_df, results_df

## Model A vs. B 

In [7]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="zhd_A_xgb_on_t2.csv",
    csv_b="zhd_B_xgb_on_t2.csv",
    model_a_name="train_A_xgb",
    model_b_name="train_B_xgb",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.321111,0.339031,0.017920,0.001094,0.034627,0.037992,5000,train_A_xgb,train_B_xgb,10066
1,auc,0.854518,0.864225,0.009707,0.003367,0.016098,0.002400,5000,train_A_xgb,train_B_xgb,10066
2,f1_05,0.189313,0.145215,-0.044098,-0.072758,-0.015881,0.002000,5000,train_A_xgb,train_B_xgb,10066
3,precision_05,0.508197,0.602740,0.094543,0.006639,0.183599,0.036793,5000,train_A_xgb,train_B_xgb,10066
4,recall_05,0.116323,0.082552,-0.033771,-0.052533,-0.015009,0.000800,5000,train_A_xgb,train_B_xgb,10066
5,mcc_05,0.225177,0.209840,-0.015337,-0.051054,0.019651,0.389522,5000,train_A_xgb,train_B_xgb,10066
6,f1_opt,0.329227,0.383775,0.054548,0.023198,0.087954,0.000800,5000,train_A_xgb,train_B_xgb,10066
7,precision_opt,0.408333,0.328438,-0.079895,-0.116476,-0.041916,0.000400,5000,train_A_xgb,train_B_xgb,10066
8,recall_opt,0.275797,0.461538,0.185741,0.151970,0.221388,0.000400,5000,train_A_xgb,train_B_xgb,10066
9,mcc_opt,0.305637,0.348805,0.043168,0.010286,0.077224,0.009598,5000,train_A_xgb,train_B_xgb,10066


In [8]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="zhd_A_lgbm_on_t2.csv",
    csv_b="zhd_B_lgbm_on_t2.csv",
    model_a_name="train_A_lgbm",
    model_b_name="train_B_lgbm",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.333322,0.340958,0.007636,-0.010921,0.027002,0.411918,5000,train_A_lgbm,train_B_lgbm,10066
1,auc,0.859163,0.867065,0.007902,0.000988,0.014873,0.020396,5000,train_A_lgbm,train_B_lgbm,10066
2,f1_05,0.192192,0.000000,-0.192192,-0.232426,-0.154321,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
3,precision_05,0.481203,0.000000,-0.481203,-0.562055,-0.401514,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
4,recall_05,0.120075,0.000000,-0.120075,-0.148218,-0.095685,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
5,mcc_05,0.221291,0.000000,-0.221291,-0.266442,-0.177760,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
6,f1_opt,0.330539,0.390060,0.059521,0.025284,0.094185,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
7,precision_opt,0.456954,0.325786,-0.131167,-0.175850,-0.087732,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
8,recall_opt,0.258912,0.485929,0.227017,0.189493,0.264540,0.000400,5000,train_A_lgbm,train_B_lgbm,10066
9,mcc_opt,0.317287,0.356780,0.039492,0.003438,0.074907,0.031994,5000,train_A_lgbm,train_B_lgbm,10066


## Model B vs. C

In [11]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="leave_B_xgb_on_t2.csv",
    csv_b="leave_C_xgb_on_t2.csv",
    model_a_name="train_B_xgb",
    model_b_name="train_C_xgb",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.454380,0.486999,0.032618,0.004233,0.062702,0.026395,5000,train_B_xgb,train_C_xgb,8748
1,auc,0.888985,0.899383,0.010398,0.004301,0.016820,0.001600,5000,train_B_xgb,train_C_xgb,8748
2,f1_05,0.149915,0.342857,0.192942,0.152125,0.234927,0.000400,5000,train_B_xgb,train_C_xgb,8748
3,precision_05,0.814815,0.718563,-0.096252,-0.197030,0.009327,0.077584,5000,train_B_xgb,train_C_xgb,8748
4,recall_05,0.082552,0.225141,0.142589,0.112570,0.172608,0.000400,5000,train_B_xgb,train_C_xgb,8748
5,mcc_05,0.248390,0.383544,0.135154,0.092248,0.182680,0.000400,5000,train_B_xgb,train_C_xgb,8748
6,f1_opt,0.465468,0.492155,0.026687,0.000720,0.053981,0.043591,5000,train_B_xgb,train_C_xgb,8748
7,precision_opt,0.469466,0.439528,-0.029938,-0.056323,-0.003045,0.026795,5000,train_B_xgb,train_C_xgb,8748
8,recall_opt,0.461538,0.559099,0.097561,0.065666,0.131332,0.000400,5000,train_B_xgb,train_C_xgb,8748
9,mcc_opt,0.431119,0.458775,0.027656,0.000237,0.056560,0.047990,5000,train_B_xgb,train_C_xgb,8748


In [10]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="leave_B_lgbm_on_t2.csv",
    csv_b="leave_C_lgbm_on_t2.csv",
    model_a_name="train_B_lgbm",
    model_b_name="train_C_lgbm",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.465387,0.490990,0.025603,-0.002053,0.054400,0.070386,5000,train_B_lgbm,train_C_lgbm,8748
1,auc,0.893910,0.900679,0.006769,0.001925,0.011873,0.005199,5000,train_B_lgbm,train_C_lgbm,8748
2,f1_05,0.000000,0.299850,0.299850,0.253870,0.345029,0.000400,5000,train_B_lgbm,train_C_lgbm,8748
3,precision_05,0.000000,0.746269,0.746269,0.671997,0.817518,0.000400,5000,train_B_lgbm,train_C_lgbm,8748
4,recall_05,0.000000,0.187617,0.187617,0.155722,0.221388,0.000400,5000,train_B_lgbm,train_C_lgbm,8748
5,mcc_05,0.000000,0.357353,0.357353,0.312737,0.400162,0.000400,5000,train_B_lgbm,train_C_lgbm,8748
6,f1_opt,0.487759,0.496097,0.008338,-0.014401,0.031654,0.481904,5000,train_B_lgbm,train_C_lgbm,8748
7,precision_opt,0.489603,0.461290,-0.028313,-0.053475,-0.003334,0.025195,5000,train_B_lgbm,train_C_lgbm,8748
8,recall_opt,0.485929,0.536585,0.050657,0.022514,0.078799,0.000800,5000,train_B_lgbm,train_C_lgbm,8748
9,mcc_opt,0.454661,0.462273,0.007612,-0.016759,0.032301,0.553089,5000,train_B_lgbm,train_C_lgbm,8748


## Model A vs. B for Grade 1

In [2]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="zhd_A_xgb_on_t2_gr1.csv",
    csv_b="zhd_B_xgb_on_t2_gr1.csv",
    model_a_name="train_A_xgb",
    model_b_name="train_B_xgb",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.527707,0.577570,0.049864,0.022278,0.078043,0.000400,5000,train_A_xgb,train_B_xgb,1542
1,auc,0.830725,0.875420,0.044696,0.018259,0.074842,0.000800,5000,train_A_xgb,train_B_xgb,1542
2,f1_05,0.453901,0.454545,0.000645,-0.066099,0.064012,0.983403,5000,train_A_xgb,train_B_xgb,1542
3,precision_05,0.744186,0.882353,0.138167,0.043134,0.245104,0.003199,5000,train_A_xgb,train_B_xgb,1542
4,recall_05,0.326531,0.306122,-0.020408,-0.071429,0.030612,0.603879,5000,train_A_xgb,train_B_xgb,1542
5,mcc_05,0.472535,0.503969,0.031434,-0.031627,0.095009,0.327534,5000,train_A_xgb,train_B_xgb,1542
6,f1_opt,0.535519,0.497854,-0.037665,-0.088137,0.017175,0.165967,5000,train_A_xgb,train_B_xgb,1542
7,precision_opt,0.576471,0.429630,-0.146841,-0.216452,-0.084738,0.000400,5000,train_A_xgb,train_B_xgb,1542
8,recall_opt,0.500000,0.591837,0.091837,0.040816,0.153061,0.000400,5000,train_A_xgb,train_B_xgb,1542
9,mcc_opt,0.507825,0.464813,-0.043012,-0.097022,0.012815,0.131574,5000,train_A_xgb,train_B_xgb,1542


In [3]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="zhd_A_lgbm_on_t2_gr1.csv",
    csv_b="zhd_B_lgbm_on_t2_gr1.csv",
    model_a_name="train_A_lgbm",
    model_b_name="train_B_lgbm",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.543975,0.532710,-0.011265,-0.045280,0.021852,0.511498,5000,train_A_lgbm,train_B_lgbm,1542
1,auc,0.849578,0.878353,0.028775,0.001475,0.059321,0.041592,5000,train_A_lgbm,train_B_lgbm,1542
2,f1_05,0.486842,0.000000,-0.486842,-0.582278,-0.388489,0.000400,5000,train_A_lgbm,train_B_lgbm,1542
3,precision_05,0.685185,0.000000,-0.685185,-0.800000,-0.571429,0.000400,5000,train_A_lgbm,train_B_lgbm,1542
4,recall_05,0.377551,0.000000,-0.377551,-0.479592,-0.285714,0.000400,5000,train_A_lgbm,train_B_lgbm,1542
5,mcc_05,0.485418,0.000000,-0.485418,-0.579915,-0.387537,0.000400,5000,train_A_lgbm,train_B_lgbm,1542
6,f1_opt,0.536313,0.486692,-0.049621,-0.112112,0.019226,0.146371,5000,train_A_lgbm,train_B_lgbm,1542
7,precision_opt,0.592593,0.387879,-0.204714,-0.283834,-0.131479,0.000400,5000,train_A_lgbm,train_B_lgbm,1542
8,recall_opt,0.489796,0.653061,0.163265,0.091837,0.234694,0.000400,5000,train_A_lgbm,train_B_lgbm,1542
9,mcc_opt,0.510614,0.460196,-0.050418,-0.116771,0.019491,0.146371,5000,train_A_lgbm,train_B_lgbm,1542


## Model B vs. C for Grade 1

In [5]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="leave_B_xgb_on_t2_gr1.csv",
    csv_b="leave_C_xgb_on_t2_gr1.csv",
    model_a_name="train_B_xgb",
    model_b_name="train_C_xgb",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.609581,0.643871,0.034289,-0.036259,0.103522,0.340332,5000,train_B_xgb,train_C_xgb,1449
1,auc,0.888333,0.904613,0.016280,-0.001050,0.033386,0.063587,5000,train_B_xgb,train_C_xgb,1449
2,f1_05,0.458015,0.594937,0.136921,0.050715,0.230018,0.002799,5000,train_B_xgb,train_C_xgb,1449
3,precision_05,0.909091,0.783333,-0.125758,-0.237366,-0.013325,0.027195,5000,train_B_xgb,train_C_xgb,1449
4,recall_05,0.306122,0.479592,0.173469,0.091837,0.255102,0.000400,5000,train_B_xgb,train_C_xgb,1449
5,mcc_05,0.511547,0.592357,0.080810,0.000003,0.167419,0.049990,5000,train_B_xgb,train_C_xgb,1449
6,f1_opt,0.563107,0.616114,0.053007,0.012067,0.097724,0.009598,5000,train_B_xgb,train_C_xgb,1449
7,precision_opt,0.537037,0.575221,0.038184,0.000144,0.078574,0.049990,5000,train_B_xgb,train_C_xgb,1449
8,recall_opt,0.591837,0.663265,0.071429,0.020408,0.132653,0.017197,5000,train_B_xgb,train_C_xgb,1449
9,mcc_opt,0.530484,0.587862,0.057378,0.013426,0.106027,0.009598,5000,train_B_xgb,train_C_xgb,1449


In [6]:
pair_df, results_df = run_paired_bootstrap_from_csv(
    csv_a="leave_B_lgbm_on_t2_gr1.csv",
    csv_b="leave_C_lgbm_on_t2_gr1.csv",
    model_a_name="train_B_lgbm",
    model_b_name="train_C_lgbm",
    n_boot=5000,
    stratified=True,
    seed=42
)

results_df

,metric,model_a,model_b,diff_b_minus_a,ci_low,ci_high,p_value,n_boot,model_a_name,model_b_name,n_common
0,ap,0.591537,0.616483,0.024946,-0.045328,0.100566,0.463107,5000,train_B_lgbm,train_C_lgbm,1449
1,auc,0.896082,0.901581,0.005499,-0.003172,0.014268,0.222356,5000,train_B_lgbm,train_C_lgbm,1449
2,f1_05,0.000000,0.489510,0.489510,0.385185,0.582278,0.000400,5000,train_B_lgbm,train_C_lgbm,1449
3,precision_05,0.000000,0.777778,0.777778,0.659562,0.888889,0.000400,5000,train_B_lgbm,train_C_lgbm,1449
4,recall_05,0.000000,0.357143,0.357143,0.265306,0.448980,0.000400,5000,train_B_lgbm,train_C_lgbm,1449
5,mcc_05,0.000000,0.506288,0.506288,0.407524,0.595821,0.000400,5000,train_B_lgbm,train_C_lgbm,1449
6,f1_opt,0.587156,0.653266,0.066110,0.029035,0.105016,0.000800,5000,train_B_lgbm,train_C_lgbm,1449
7,precision_opt,0.533333,0.643564,0.110231,0.064825,0.161592,0.000400,5000,train_B_lgbm,train_C_lgbm,1449
8,recall_opt,0.653061,0.663265,0.010204,-0.030612,0.061224,0.813037,5000,train_B_lgbm,train_C_lgbm,1449
9,mcc_opt,0.557266,0.627790,0.070524,0.030877,0.112675,0.001200,5000,train_B_lgbm,train_C_lgbm,1449
